# 🛰️ SAT-GUARDIAN: Interactive Demo
**Physics-Aware Adaptive Satellite Frame Interpolation System**

This notebook demonstrates the full SAT-GUARDIAN pipeline:
1. Load / generate sample satellite frames and ERA5 wind
2. Compute physics-constrained optical flow
3. Generate T0.25, T0.50, T0.75 via cascaded interpolation
4. Compute confidence maps and Cloud Motion Consistency Score
5. Evaluate quality metrics
6. Create visualisations and GIF animation


In [ ]:
import sys, os
from pathlib import Path

# Add src to path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['figure.facecolor'] = '#0f1117'
matplotlib.rcParams['text.color'] = 'white'
%matplotlib inline

print('✅ Imports successful')
print(f'Root: {ROOT}')


## Step 1: Load Sample Data

In [ ]:
from data_loader import load_sample_data

sample = load_sample_data(sample_dir=str(ROOT / 'data' / 'sample'), H=256, W=256)

frame_t0 = sample['frame_t0']
frame_t1 = sample['frame_t1']
wind_u   = sample['wind_u']
wind_v   = sample['wind_v']

print(f'T0:  shape={frame_t0.shape}, range=[{frame_t0.min():.3f}, {frame_t0.max():.3f}]')
print(f'T1:  shape={frame_t1.shape}, range=[{frame_t1.min():.3f}, {frame_t1.max():.3f}]')
print(f'U:   shape={wind_u.shape},  range=[{wind_u.min():.2f}, {wind_u.max():.2f}] m/s')
print(f'V:   shape={wind_v.shape},  range=[{wind_v.min():.2f}, {wind_v.max():.2f}] m/s')


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.patch.set_facecolor('#0f1117')
for ax in axes:
    ax.set_facecolor('#0f1117')

axes[0].imshow(frame_t0, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('T₀ (Input)', color='white', fontweight='bold')
axes[0].axis('off')

axes[1].imshow(frame_t1, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('T₁ (Input)', color='white', fontweight='bold')
axes[1].axis('off')

im2 = axes[2].imshow(wind_u, cmap='RdBu_r')
axes[2].set_title('ERA5 Wind U (m/s)', color='white', fontweight='bold')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2])

im3 = axes[3].imshow(wind_v, cmap='RdBu_r')
axes[3].set_title('ERA5 Wind V (m/s)', color='white', fontweight='bold')
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3])

plt.suptitle('Input Data Overview', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 2: Run SAT-GUARDIAN Pipeline

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

from inference import SATGuardianInference

engine  = SATGuardianInference(str(ROOT / 'configs' / 'config.yaml'))
results = engine.run(frame_t0, frame_t1, wind_u, wind_v)

print('\n✅ Generated frames:')
for key in ['frame_025', 'frame_050', 'frame_075']:
    f = results[key]
    print(f'  {key}: shape={f.shape}, range=[{f.min():.3f}, {f.max():.3f}]')


## Step 3: Visualise Generated Frames

In [ ]:
from visualization import plot_frame_comparison

fig = plot_frame_comparison(
    results['frame_t0'], results['frame_025'],
    results['frame_050'], results['frame_075'],
    results['frame_t1'],
)
plt.show()


## Step 4: Optical Flow Analysis

In [ ]:
from visualization import plot_flow_overlay

fig = plot_flow_overlay(results['frame_t0'], results['flow_fwd'])
plt.show()


## Step 5: Confidence Map

In [ ]:
from confidence_map import confidence_stats
from visualization import plot_confidence_heatmap

stats = confidence_stats(results['confidence_050'])
print('Confidence Stats:')
for k, v in stats.items():
    print(f'  {k:15s}: {v:.4f}')

fig = plot_confidence_heatmap(results['confidence_050'], frame=results['frame_050'])
plt.show()


## Step 6: Cloud Motion Score & Metrics

In [ ]:
from metrics import evaluate_frame, temporal_consistency_score

cm = results['cloud_motion']
print(f'Cloud Motion Score : {cm["overall_score"]:.1f}/100')
print(f'Interpretation     : {cm["interpretation"]}')

# Quality metrics vs linear interpolation GT
gt_050 = 0.5 * frame_t0 + 0.5 * frame_t1
m = evaluate_frame(results['frame_050'], gt_050, 'T0.50 vs Linear GT')
print(f"\nSSIM: {m['ssim']:.4f} | PSNR: {m['psnr']:.2f} dB | MSE: {m['mse']:.6f} | FSIM: {m['fsim']:.4f}")

seq = [results['frame_t0'], results['frame_025'], results['frame_050'],
       results['frame_075'], results['frame_t1']]
tc = temporal_consistency_score(seq)
print(f'Temporal Consistency: {tc:.4f}')


## Step 7: Save All Outputs & Dashboard

In [ ]:
from metrics import evaluate_frame

metrics_list = [
    evaluate_frame(results['frame_025'], 0.75*frame_t0+0.25*frame_t1, 'T0.25'),
    evaluate_frame(results['frame_050'], 0.50*frame_t0+0.50*frame_t1, 'T0.50'),
    evaluate_frame(results['frame_075'], 0.25*frame_t0+0.75*frame_t1, 'T0.75'),
]

engine.save_all(results, output_dir=str(ROOT / 'outputs'), metrics_list=metrics_list)
print('\n✅ All outputs saved to outputs/')
print('Check: frame_comparison.png, dashboard.png, animations/interpolation.gif')


## Step 8: Display Dashboard


In [ ]:
from visualization import plot_dashboard
import matplotlib.pyplot as plt

fig = plot_dashboard(
    results['frame_t0'], results['frame_025'], results['frame_050'],
    results['frame_075'], results['frame_t1'],
    results['confidence_050'],
    results['flow_fwd'],
    results['cloud_motion']['overall_score'],
    metrics_list=metrics_list,
)
plt.show()
